In [1]:
from openai import OpenAI
from dotenv import load_dotenv
from pathlib import Path
import os
import json
from IPython.display import Markdown

load_dotenv(override=True)

True

In [ ]:
llm_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPEN_ROUTER_KEY"),
)

# llm_client = OpenAI(
#     base_url="https://api.groq.com/openai/v1",
#     api_key=os.getenv("GROQ_API_KEY"),
# )

MODEL = "nvidia/nemotron-3.5-lightning:free"       # chat model (free, NVIDIA)
# MODEL = "openai/gpt-oss-120b"

In [3]:
menu = [
    {"name": "Zinger Burger", "price": 850},
    {"name": "Chicken Biryani", "price": 450},
    {"name": "Club Sandwich", "price": 700}
]

complaint_list = []

order_list = [
    "OR-1045",
    "OR-1046",
    "OR-1047"
]

In [4]:
def fetch_menu():
    return menu

def log_complaint(customer_name, order_id, phone, complaint):
    temp_dict = {
        "customer_name": customer_name,
        "order_id": order_id,
        "phone": phone,
        "complaint": complaint,
        "status": "Pending"
    }
    complaint_list.append(temp_dict)

    return {
        "message": f"Saved! Total Complaint: {len(complaint_list)}"
    }

def validate_order_id(order_id):
    if order_id in order_list:
        return True
    return False


tools = [
    {
        "type": "function",
        "function": {
                "name": "fetch_menu",
                "description": "Get the restaurant menu",
                # no inputs
                "parameters": {"type": "object", "properties": {}},
        },
    },
    {
        "type": "function",
        "function": {
            "name": "validate_order_id",
            "description": "Validate order id before calling log_complaint tool it return True/False",
            "parameters": {
                "type": "object", 
                "properties": {
                    "order_id": {
                        "type": "string",
                        "description": "the order id of which customer ordered food. OrderId should be like this OR-0000",
                    },
                },
                "required": ["order_id"],
                "additionalProperties": False,

            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "log_complaint",
            "description": "Log the complain",
            "strict": True,
            "parameters": {
                "type": "object", 
                "properties": {
                    "customer_name": {
                        "type": "string",
                        "description": "Customer name who is logging complain.",
                    },
                    "order_id": {
                        "type": "string",
                        "description": "the order id of which customer ordered food. OrderId should be like this OR-0000",
                    },
                    "phone": {
                        "type": "string",
                        "description": "Customer phone number or '-' ",
                    },
                    "complaint": {
                        "type": "string",
                        "description": "customer complain should me not more than 8 words",
                    }
                },
                "required": ["customer_name", "order_id", "phone", "complaint"],
                "additionalProperties": False,

            },
        }
    }

]

In [5]:
messages = [
    {
      "role": "system",  # User | Assistant | System
      "content": """
       You are the AI Customer Support Assistant for **Xander Restaurant**. Your job is to answer customer queries in a friendly, professional, and helpful way.

        ### Tools
        1. **`fetch_menu`** — Call whenever customers ask about menu items, prices, ingredients, or food recommendations. Use the tool's response to provide accurate information.
        2. **`log_complaint`** — Call whenever a customer raises a complaint. Collect the necessary details and log the complaint using this schema:

        ```python
        complaint = {
            "customer_name": "",
            "phone": "",
            "complaint": "",
            "order_id": "",
            "status": "Pending"
        }
        ```

        3. **`validate_order_id`** — Before logging complaint trigger this tool call validate_order_id to validate wether the order id is valid or not then call log_complaint


        Ask for missing necessary details before logging. The `order_id` can be empty if unavailable. Set the initial status to `"Pending"`. Confirm the complaint is logged only after the tool succeeds.

        ### Guidelines
        - Keep replies short, natural, and friendly.
        - Never invent menu details, prices, or restaurant policies.
        - If you don't know something, politely say so.
        - Make every customer feel valued.
        - All prices are in Pakistani Rupees. Write them as "Rs 850".
      """
    }
]


In [ ]:
function_names = {
    "fetch_menu": fetch_menu,
    "log_complaint": log_complaint,
    "validate_order_id": validate_order_id
}

In [ ]:
def ask_assistant(question):
    messages.append({
        "role": "user", 
        "content": question
    })

    while True:
        completion = llm_client.chat.completions.create(
                model=MODEL, 
                messages=messages, 
                tools=tools,
                temperature=0
            )
           
        reply = completion.choices[0].message
        messages.append(reply)
    
        # reply_json = reply.model_dump(exclude_none=True)
        # reply_json.pop("reasoning", None)            # hide the model's long thinking
        # reply_json.pop("reasoning_details", None)
        # print("LLM REPLY:", json.dumps(reply_json, indent=2))
    
        if not reply.tool_calls: 
            return Markdown(reply.content)
            break                        # model asked for a tool
       
        for call in reply.tool_calls:
            args = json.loads(call.function.arguments or "{}")
            result = function_names[call.function.name](**args)     # run our Python function
            print("Tool called:", call.function.name, args, result)
            messages.append({"role": "tool", "tool_call_id": call.id, "content": json.dumps(result)})        

In [7]:
ask_assistant("i ordered burger but roll was served WTH!!! and the bill was made with burger, my name is Sam, OR-1045, phone is 0322342423")

Tool called: validate_order_id {'order_id': 'OR-1045'} True
Tool called: log_complaint {'complaint': 'Received roll instead of burger', 'customer_name': 'Sam', 'order_id': 'OR-1045', 'phone': '0322342423'} {'message': 'Saved! Total Complaint: 1'}


Your complaint has been logged, Sam. We’re sorry for the mix‑up with your order. Our team will review it and get back to you shortly. Thank you for letting us know!

In [8]:
print(json.dumps(complaint_list))

[{"customer_name": "Sam", "order_id": "OR-1045", "phone": "0322342423", "complaint": "Received roll instead of burger", "status": "Pending"}]




Create a new tool validate_order to validate orderId when logging complaint
if order id does not gets validated say user please provider correct Order ID

In [11]:
import pypandoc
pypandoc.convert_file("Rag Example/xander_restaurant_knowledge_base.md", "plain", outputfile="Rag Example/plain.txt")

''

In [13]:
file_path = Path("Rag Example/plain.txt")
text = file_path.read_text(encoding="utf-8")
print(text)


print(f"{file_path.name} has {len(text)} characters")
print(f"{file_path.name} has {len(text.split())} words")

Xander Restaurant — Knowledge Base

Document type: Synthetic demo data for RAG testing
Version: 1.0
Important: Xander Restaurant and all details in this document are
fictional sample data created for testing. Replace them with verified
business information before using this knowledge base with real
customers.

------------------------------------------------------------------------

1. Restaurant Overview

Name: Xander Restaurant
Cuisine: Casual dining with a mix of burgers, sandwiches, rice dishes,
grilled items, and local favourites
Positioning: A friendly, modern restaurant focused on freshly prepared
meals, approachable prices, and convenient dining.
Service options: Dine-in, takeaway, and delivery (availability may vary
by branch).
Currency: Pakistani Rupees (PKR / Rs.)

Xander Restaurant aims to make everyday dining convenient for families,
students, office workers, and groups of friends. Each branch follows the
same core service standards, although some dishes are exclusive to
s

In [ ]:
EMB_MODEL = "nvidia/nemotron-3-embed-1b:free" 

client = llm_client
def embed(texts):
    r = client.embeddings.create(model=EMB_MODEL, input = texts)
    vectors = []
    for items in r.data:
        vectors.append(items) 
        vectors.append(items.embedding) 
    return vectors

print(embed(text))


        

NotFoundError: Error code: 404 - The model `liquid/lfm-2.5-embedding-350m:free` does not exist or you do not have access to it.

In [27]:
def chunk_by_paragraph(text):
    parts = []
    for p in text.split("\n\n"):
        raw = p.strip()
        parts.append(raw)
    chunks = []
    for p in parts:
        if len(p)>30:
            chunks.append(p)
    return chunks
# i = 0
chunked_text = chunk_by_paragraph(text)
for chunk in chunked_text:
    # i = i + 1
    print("-",chunk)

print(f"Count of chunked sentences : {len(chunked_text)}")
# print(i)
        
           
        
    

- Xander Restaurant — Knowledge Base
- Document type: Synthetic demo data for RAG testing
Version: 1.0
Important: Xander Restaurant and all details in this document are
fictional sample data created for testing. Replace them with verified
business information before using this knowledge base with real
customers.
- ------------------------------------------------------------------------
- Name: Xander Restaurant
Cuisine: Casual dining with a mix of burgers, sandwiches, rice dishes,
grilled items, and local favourites
Positioning: A friendly, modern restaurant focused on freshly prepared
meals, approachable prices, and convenient dining.
Service options: Dine-in, takeaway, and delivery (availability may vary
by branch).
Currency: Pakistani Rupees (PKR / Rs.)
- Xander Restaurant aims to make everyday dining convenient for families,
students, office workers, and groups of friends. Each branch follows the
same core service standards, although some dishes are exclusive to
selected locations.

In [32]:
import pandas as pd

pd.DataFrame(chunked_text)


,0
0,Xander Restaurant — Knowledge Base
1,Document type: Synthetic demo data for RAG tes...
2,----------------------------------------------...
3,Name: Xander Restaurant\nCuisine: Casual dinin...
4,Xander Restaurant aims to make everyday dining...
5,Xander Restaurant was founded in 2018 by Ayaan...
6,The idea began when the founders noticed that ...
7,"The first location opened in Gulshan-e-Iqbal, ..."
8,Founding facts for retrieval - Founded: 2018 -...
9,Branch 1 — Gulshan-e-Iqbal (Flagship)
